# 04 — I still don't see it — show me another way

**Optional visual-intuition companion / workbench**

This notebook does not extend Lambert et al.'s claims. Its repeated rhythm is:

**👀 see → 🧹 remove information → 🔍 isolate → 📏 collapse → 🧠 rebuild → ⚠️ stop where the evidence stops**

> Figures 5, 6, and 8 are fixed author-binned images, not source stars. Figure 13 contains rows from an already-selected sample. Figure 9 contains binned summaries and Figure 10 an author-derived spectrum. None is the unpublished DESI DR2 catalogue.

In [ ]:
from pathlib import Path

import astropy.units as u
import matplotlib.pyplot as plt
import numpy as np
from astropy.table import Table
from matplotlib.colors import BoundaryNorm, ListedColormap, LogNorm
from matplotlib.patches import Patch
from scipy.signal import savgol_filter

from lambert_lab.data import (
    load_figure10_spectrum, load_figure13_stars, load_figure9_wave,
    load_lambert_inventory, load_released_image,
)
from lambert_lab.plotting import (
    SIGN_CATEGORY_LABELS, binned_median_visual_guide, constant_lz_curves,
    display_pixel_centers, display_released_image, positive_run_containing_maximum,
    raw_column_peak, robust_symmetric_limit,
    released_pixel_profile, toy_cycle_spectrum, velocity_sign_categories,
    zero_centered_norm,
)
from lambert_lab.spectral import spectral_resolution

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
inventory = load_lambert_inventory(ROOT)
print("Lambert release:", inventory["release"]["doi"])
print("No network access; no source-level remeasurement.")

## Reading key

Each derived panel is either **our display transformation** or **our pedagogical guide**. The fixed images have shapes and header-stated extents but no exact WCS/bin edges. Coordinates made by evenly spacing pixel centres across an extent are therefore called **nominal display coordinates**, never recovered physical bin edges.

# Figure 5 — First see direction, then amplitude

## 👀 What am I supposed to see?

$V_R>0$ means outward; $V_R<0$ inward. $V_Z>0$ means toward the North Galactic Pole; $V_Z<0$ downward.

## 🧹 Remove information: geometry only

The locator below contains no velocity data. The Monoceros/ACS labels are approximate regions, not released boundaries.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.8))
ax.set(xlim=(150, 220), ylim=(20, 40), xlabel=r"Galactic longitude $l$ [deg]",
       ylabel=r"Galactic latitude $b$ [deg]", title="Geometry-only Figure 5 canvas")
ax.axvline(180, color="0.25", ls="--")
ax.text(181, 20.7, r"anticentre longitude; $b=0^\circ$ is off-panel", fontsize=9)
ax.annotate("lower-b Monoceros-associated\nregion (approximate)",
            (208, 26), (195, 22), arrowprops={"arrowstyle": "->"}, ha="center")
ax.annotate("higher-b ACS-associated\nregion (approximate; no boundary)",
            (175, 36.5), (158, 33), arrowprops={"arrowstyle": "->"}, ha="center")
ax.grid(alpha=0.2)
fig.tight_layout()
plt.show()

## 🔍 Isolate one feature: strict signs

These are transformations of released author-bin medians. Missing values and exact zeros remain visibly undefined. The categorical panel shows four combinations of released median-velocity pixel signs—not four stellar populations or population fractions.

In [ ]:
fig5_vr = load_released_image(ROOT, "lb_VR_fig5.fits")
fig5_vz = load_released_image(ROOT, "lb_VZ_fig5.fits")
sign_cmap = ListedColormap(["#3b4cc0", "#b40426"])
sign_cmap.set_bad(color="0.82")
state_colors = ["#3b4cc0", "#55a868", "#dd8452", "#b40426"]
states5 = velocity_sign_categories(fig5_vr.data, fig5_vz.data)
state_cmap = ListedColormap(state_colors)
state_cmap.set_bad(color="0.82")
longitude5_sign, _ = display_pixel_centers(fig5_vr)
sign_strip = longitude5_sign[[15, 20]]

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2), sharex=True, sharey=True)
for ax, product, title, labels in [
    (axes[0], fig5_vr, r"$V_R$ sign", ("inward", "outward")),
    (axes[1], fig5_vz, r"$V_Z$ sign", ("downward", "upward")),
]:
    valid = np.isfinite(product.data) & (product.data != 0)
    signs = np.ma.array((product.data > 0).astype(int), mask=~valid)
    ax.imshow(signs, origin="lower", extent=product.extent, aspect="auto",
              interpolation="nearest", cmap=sign_cmap,
              norm=BoundaryNorm([-0.5, 0.5, 1.5], 2))
    ax.axvspan(*sign_strip, facecolor="none", edgecolor="black", lw=1.1)
    ax.set(title=title, xlabel=r"$l$ [deg]", ylabel=r"$b$ [deg]")
    ax.legend(handles=[Patch(color=sign_cmap(i), label=label)
                       for i, label in enumerate(labels)], fontsize=8)
axes[2].imshow(states5, origin="lower", extent=fig5_vr.extent, aspect="auto",
               interpolation="nearest", cmap=state_cmap,
               norm=BoundaryNorm(np.arange(-0.5, 4.5), 4))
axes[2].axvspan(*sign_strip, facecolor="none", edgecolor="black", lw=1.1)
axes[2].set(title="four combinations of released median-velocity pixel signs", xlabel=r"$l$ [deg]", ylabel=r"$b$ [deg]")
axes[2].legend(handles=[Patch(color=state_colors[k], label=v)
                        for k, v in SIGN_CATEGORY_LABELS.items()], fontsize=7)
fig.tight_layout()
plt.show()

## 📏 Collapse it: a latitude dissection

Zero-based columns **15–20** have nominal centres $175.83^\circ$–$184.17^\circ$, a symmetric nominal footprint $175^\circ$–$185^\circ$. Thin lines are individual released pixels; thick lines are unweighted finite means of six author-bin values—not source-level stellar medians.

In [ ]:
FIG5_COLUMNS = np.arange(15, 21)
longitude5, latitude5 = display_pixel_centers(fig5_vr)
b_vr, profile_vr = released_pixel_profile(fig5_vr, collapse_axis=1, indices=FIG5_COLUMNS)
b_vz, profile_vz = released_pixel_profile(fig5_vz, collapse_axis=1, indices=FIG5_COLUMNS)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.3), sharex=True)
for ax, product, profile, symbol in [
    (axes[0], fig5_vr, profile_vr, r"$V_R$"),
    (axes[1], fig5_vz, profile_vz, r"$V_Z$"),
]:
    for column in FIG5_COLUMNS:
        ax.plot(latitude5, product.data[:, column], color="0.72", lw=0.8)
    ax.plot(latitude5, profile, "o-", color="black", lw=2,
            label="mean of six released pixels")
    ax.axhline(0, color="0.35", lw=1)
    ax.set(xlabel=r"nominal display $b$ [deg]",
           ylabel=fr"{symbol} [km s$^{{-1}}$]", title=fr"{symbol}(b)$")
    ax.legend(fontsize=8)
fig.tight_layout()
plt.show()
print("columns:", FIG5_COLUMNS.tolist())
print("nominal l centres:", np.round(longitude5[FIG5_COLUMNS], 3))

## 🧠 Rebuild the interpretation

Lower latitude is predominantly inward/downward. Higher latitude is more mixed and contains coherent outward/upward patches; the profile also preserves real reversals and longitude variation. There is no hard latitude boundary. Returning to the continuous maps restores amplitude.

## ⚠️ What this still does not prove

Sign maps discard magnitude and uncertainty. Image uncertainties, source counts, exact edges, and the ACS photometric boundary are absent. Nothing here reconstructs author medians, membership, or a formation mechanism.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), sharex=True, sharey=True)
for ax, product, symbol in [(axes[0], fig5_vr, r"$V_R$"), (axes[1], fig5_vz, r"$V_Z$")]:
    image = display_released_image(ax, product, cmap="coolwarm",
        norm=zero_centered_norm(product.data), title="released median " + symbol)
    ax.axvspan(175, 185, facecolor="none", edgecolor="black", lw=1.4)
    fig.colorbar(image, ax=ax, label=symbol + r" [km s$^{-1}$]")
fig.tight_layout()
plt.show()

# Figure 6 — Density answers a different question

## 👀 What am I supposed to see?

- Density residual: **Where are there more/fewer stars than the fitted smooth model?**
- $V_R$: **Where does the selected population share radial motion?**
- $V_Z$: **Where does it share vertical motion?**

## 🧹 Remove information

Read each map separately. The residual is already relative to the authors' exponential-disc model, not a raw count.

In [ ]:
names6 = ["XY_overdensity_fig6.fits", "XY_VR_fig6.fits", "XY_VZ_fig6.fits"]
products6 = [load_released_image(ROOT, name) for name in names6]
questions6 = [
    "Where is the released density\nresidual positive or negative?",
    "Which way is the median\nradial motion?",
    "Which way is the median\nvertical motion?",
]
units6 = ["effective count", r"km s$^{-1}$", r"km s$^{-1}$"]
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2), sharex=True, sharey=True)
for ax, product, question, unit in zip(axes, products6, questions6, units6):
    image = display_released_image(ax, product, cmap="coolwarm",
        norm=zero_centered_norm(product.data), title=question)
    ax.axhspan(-1, 1, facecolor="none", edgecolor="black", lw=1.2)
    fig.colorbar(image, ax=ax, label=unit)
fig.tight_layout()
plt.show()

## 🔍 Isolate and 📏 collapse

Average the identical predeclared released rows **7 and 8**, with nominal $Y=-0.5,+0.5$ kpc: the nominal display strip $-1<Y<1$ kpc. Native units and separate vertical scales are retained. The single black contour is only a display overlay of one author product on another.

In [ ]:
FIG6_ROWS = [7, 8]
profiles6 = [released_pixel_profile(p, collapse_axis=0, indices=FIG6_ROWS)
             for p in products6]
for coordinate, _ in profiles6[1:]:
    np.testing.assert_array_equal(profiles6[0][0], coordinate)
positive_start6, positive_stop6 = positive_run_containing_maximum(profiles6[0][1])
positive_interval6 = profiles6[0][0][[positive_start6, positive_stop6 - 1]]
print("Main positive density-residual run, nominal display X [kpc]:", positive_interval6)

fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
labels6 = ["density residual [effective count]", r"median $V_R$ [km s$^{-1}$]",
           r"median $V_Z$ [km s$^{-1}$]"]
for ax, (coordinate, profile), label in zip(axes, profiles6, labels6):
    ax.axvspan(*positive_interval6, color="tab:orange", alpha=0.14,
               label="main positive density-residual pixel run")
    ax.plot(coordinate, profile, "o-", ms=4)
    ax.axhline(0, color="0.35", lw=1)
    ax.set_ylabel(label)
axes[-1].set_xlabel(r"nominal display $X$ [kpc]")
axes[0].set_title(r"Profiles through nominal $-1<Y<1$ kpc")
axes[0].legend(fontsize=8)
fig.tight_layout()
plt.show()

density6, vr6, vz6 = products6
x6, y6 = display_pixel_centers(density6)
fig, ax = plt.subplots(figsize=(7.5, 5))
image = display_released_image(ax, vr6, cmap="coolwarm",
    norm=zero_centered_norm(vr6.data), title=r"$V_R$ plus residual-zero contour")
ax.contour(x6, y6, density6.data, levels=[0], colors="0.45",
           linestyles="--", linewidths=0.7)
ax.plot([], [], color="0.45", ls="--", lw=0.7,
        label="density residual = 0 display contour — not an MRi boundary")
ax.legend(fontsize=8)
fig.colorbar(image, ax=ax, label=r"$V_R$ [km s$^{-1}$]")
fig.tight_layout()
plt.show()

## 🧠 Rebuild the interpretation

Along the central strip, density changes from deficit to excess after the velocity fields have already become negative, and negative velocity structure persists beyond much of the positive residual. Density morphology and dynamical morphology overlap only partially.

## ⚠️ What this still does not prove

The density model and $\leq5$-star mask are embedded. Profiles average author pixels, not stars; exact edges, uncertainties, and source counts are absent. A 1-D cut misses off-axis structure, and the contour is not a new measurement.

# Figure 8 — A ridge is shifted one-dimensional profiles

## 👀 What am I supposed to see?

Circular-orbit intuition is roughly horizontal in $R$–$V_\phi$. Constant angular momentum bends as $V_\phi=L_Z/R$. Both are guides, not fits.

## 🧹 Remove information

The horizontal cartoon uses the project's Lambert reference speed, $V_{c,\mathrm{ref}}=239.26$ km s$^{-1}$, explicitly as a reference—not a prediction or fitted circular speed.

In [ ]:
V_C_REF = 239.26
radius8 = np.linspace(9, 24, 300)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.3), sharex=True, sharey=True)
axes[0].axhline(V_C_REF, color="black", lw=2)
axes[0].text(10, V_C_REF+5, r"$V_\phi\simeq V_{c,\rm ref}$"
             + "\ncartoon/reference only", fontsize=9)
axes[0].set_title("Boring circular-orbit intuition")
curve3000 = constant_lz_curves(radius8, [3000])[3000.0]
axes[1].plot(radius8, curve3000, color="tab:purple", lw=2,
             label=r"$L_Z=3000$: $V_\phi=L_Z/R$")
points_r = np.array([12., 15., 20.])
axes[1].scatter(points_r, 3000/points_r, color="black")
for r, v in zip(points_r, 3000/points_r):
    axes[1].annotate(f"({r:.0f}, {v:.0f})", (r, v), xytext=(4, 5),
                     textcoords="offset points", fontsize=8)
axes[1].legend(fontsize=8)
axes[1].set_title("One constant-$L_Z$ guide")
for ax in axes:
    ax.set(xlim=(9,24), ylim=(140,270), xlabel=r"$R$ [kpc]",
           ylabel=r"$V_\phi$ [km s$^{-1}$]")
fig.tight_layout()
plt.show()

## 🔍 Isolate the guide geometry, then reveal the released density

Dashed curves at $L_Z=2500,3000,3500$ kpc km s$^{-1}$ are ours. Log colour scaling changes contrast only.

In [ ]:
fig8 = load_released_image(ROOT, "RVphi_count_fig8.fits")
finite8 = fig8.data[np.isfinite(fig8.data) & (fig8.data > 0)]
guides8 = constant_lz_curves(radius8, [2500, 3000, 3500])
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), sharex=True, sharey=True)
for value, curve in guides8.items():
    for ax in axes:
        ax.plot(radius8, curve, "--", lw=1.3, label=fr"our guide: $L_Z={value:.0f}$")
axes[0].set_title("Guides alone")
axes[0].legend(fontsize=8)
image = display_released_image(axes[1], fig8, cmap="viridis",
    norm=LogNorm(finite8.min(), finite8.max()), title="Released density plus guides")
fig.colorbar(image, ax=axes[1], label="completeness-corrected density")
for ax in axes:
    ax.set(xlim=(9,24), ylim=(140,270))
fig.tight_layout()
plt.show()

## 📏 Collapse it: four predeclared columns

Use zero-based columns **3, 9, 15, 21**, evenly separated by six pixels. Raw density-map values are unsmoothed. Open circles are guarded raw argmax markers—not centroids, uncertainties, connected tracks, or a ridge fit.

In [ ]:
FIG8_COLUMNS = [3, 9, 15, 21]
r8, vphi8 = display_pixel_centers(fig8)
colors8 = plt.cm.plasma(np.linspace(0.1, 0.85, 4))
peak_rows = []
fig = plt.figure(figsize=(14, 8))
grid = fig.add_gridspec(2, 4, height_ratios=[1, 1.3])
for position, (column, color) in enumerate(zip(FIG8_COLUMNS, colors8)):
    profile = fig8.data[:, column]
    peak_index, peak_vphi, peak_count = raw_column_peak(fig8, column)
    finite_count = int(np.isfinite(profile).sum())
    ax = fig.add_subplot(grid[0, position])
    ax.plot(profile, vphi8, "o-", ms=3, color=color)
    ax.scatter(peak_count, peak_vphi, s=70, facecolors="none",
               edgecolors="black", linewidths=1.4)
    support_note = "; sparser released support" if column == FIG8_COLUMNS[-1] else ""
    ax.set(title=f"column {column}; nominal R={r8[column]:.2f} kpc\n"
                 f"{finite_count} finite released V_phi pixels{support_note}",
           xlabel="released density-map value", ylabel=r"nominal $V_\phi$ [km s$^{-1}$]")
    peak_rows.append((column, r8[column], finite_count, peak_vphi, peak_count))

ax = fig.add_subplot(grid[1, :])
image = display_released_image(ax, fig8, cmap="viridis",
    norm=LogNorm(finite8.min(), finite8.max()), title="Raw-argmax markers on released map")
ax.text(0.01, 0.98,
        "Our open circles: raw column maxima for visual anatomy.\n"
        "Lambert's published white circles: median V_phi at each R.\n"
        "These are not the same statistic.",
        transform=ax.transAxes, va="top", fontsize=8,
        bbox={"facecolor": "white", "alpha": 0.78, "edgecolor": "none"})
for row, color in zip(peak_rows, colors8):
    ax.axvline(row[1], color=color, lw=1, alpha=0.7)
    ax.scatter(row[1], row[3], s=100, facecolors="none",
               edgecolors=color, linewidths=2)
fig.colorbar(image, ax=ax, label="completeness-corrected density")
fig.tight_layout()
plt.show()
peak_table8 = Table(rows=peak_rows, names=[
    "column", "nominal R [kpc]", "finite pixels",
    "raw-argmax nominal V_phi [km/s]", "released density-map value"])
peak_table8

## 🧠 Rebuild the interpretation

The strongest profile region shifts overall toward lower $V_\phi$ with increasing $R$, though the middle profiles are broad and the outer profile has less support. Shifted 1-D peaks are the visual origin of a 2-D ridge. Constant-$L_Z$ is useful language only after seeing that structure.

## ⚠️ What this still does not prove

Markers have no uncertainties and are not a fit. Exact $V_\phi$ edges and upstream counts are unavailable. Similarity to guides proves neither a single angular momentum nor tidal/Sagittarius causality or timing.

# Figure 13 — Condition one slice before comparing five

## 👀 What am I supposed to see?

The feature is a **Galactic-latitude–vertical-velocity ($b$–$V_Z$) correlation**, or sky-position/vertical-velocity sequence, after conditioning on $V_R$. Galactic latitude $b$ is not physical height $Z$.

## 🧹 Remove information

Predeclare $170^\circ\leq l<180^\circ$. It was not selected for the largest correlation. First show all rows, then colour those same rows by $V_R$. The colour scale uses the 95th percentile of $|V_R|$ in this displayed slice: all rows remain plotted, while more extreme values are saturated for display only.

In [ ]:
stars13 = load_figure13_stars(ROOT)
central13 = (stars13["l"] >= 170*u.deg) & (stars13["l"] < 180*u.deg)
outward13 = central13 & (stars13["V_R"] > 0*u.km/u.s)
inward13 = central13 & (stars13["V_R"] < 0*u.km/u.s)
counts13 = tuple(int(mask.sum()) for mask in (central13, outward13, inward13))
print("central slice (all, outward, inward):", counts13)
b13 = stars13["b"].to_value(u.deg)
vz13 = stars13["V_Z"].to_value(u.km/u.s)
vr13 = stars13["V_R"].to_value(u.km/u.s)
limit13 = robust_symmetric_limit(vr13[central13], percentile=95)
print(f"display-only symmetric V_R limit (95th percentile): +/-{limit13:.2f} km/s")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharex=True, sharey=True)
axes[0].scatter(b13[central13], vz13[central13], s=7, alpha=0.35,
                color="0.25", linewidths=0, rasterized=True)
axes[0].set_title("All 1,228 released rows")
points = axes[1].scatter(b13[central13], vz13[central13], c=vr13[central13],
    s=7, alpha=0.5, cmap="coolwarm", norm=zero_centered_norm(vr13[central13], limit=limit13),
    linewidths=0, rasterized=True)
axes[1].set_title(r"Same rows coloured by $V_R$")
fig.colorbar(points, ax=axes[1], extend="both",
             label=r"$V_R$ [km s$^{-1}$]; beyond limit saturated for display")
for ax in axes:
    ax.axhline(0, color="0.35", lw=0.8)
    ax.set(xlabel=r"$b$ [deg]", ylabel=r"$V_Z$ [km s$^{-1}$]",
           xlim=(20,40), ylim=(-180,180))
fig.tight_layout()
plt.show()

## 🔍 Isolate and 📏 collapse

For both sign subsets, compute medians in the same fixed 2° latitude bins and display only bins with at least 20 rows. These are **our visual guides — not physical fits**—not regressions or significance claims.

In [ ]:
guide13 = binned_median_visual_guide(
    b13[outward13], vz13[outward13], np.arange(20, 42, 2), min_count=20)
guide13_inward = binned_median_visual_guide(
    b13[inward13], vz13[inward13], np.arange(20, 42, 2), min_count=20)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6), sharex=True, sharey=True)
axes[0].scatter(b13[outward13], vz13[outward13], s=8, alpha=0.35,
                color="tab:red", linewidths=0, rasterized=True)
axes[0].plot(guide13.centers[guide13.included], guide13.medians[guide13.included],
             "o-", color="black", lw=2, label="our visual guide — not a physical fit")
axes[0].legend(fontsize=8)
axes[0].set_title(r"$V_R>0$: outward rows")
axes[1].scatter(b13[inward13], vz13[inward13], s=8, alpha=0.35,
                color="tab:blue", linewidths=0, rasterized=True)
axes[1].plot(guide13_inward.centers[guide13_inward.included],
             guide13_inward.medians[guide13_inward.included],
             "o-", color="black", lw=2, label="our visual guide — not a physical fit")
axes[1].legend(fontsize=8)
axes[1].set_title(r"$V_R<0$: inward rows")
for ax in axes:
    ax.axhline(0, color="0.35", lw=0.8)
    ax.set(xlabel=r"$b$ [deg]", ylabel=r"$V_Z$ [km s$^{-1}$]",
           xlim=(20,40), ylim=(-180,180))
fig.tight_layout()
plt.show()
guide_table13 = Table(
    [guide13.centers, guide13.counts, guide13_inward.counts,
     guide13.medians, guide13_inward.medians, guide13.included, guide13_inward.included],
    names=["b-bin centre [deg]", "outward rows", "inward rows",
           "outward median V_Z [km/s]", "inward median V_Z [km/s]",
           "outward guide shown", "inward guide shown"])
guide_table13

## 🧠 Rebuild the interpretation

The outward subset shows an organized $b$–$V_Z$ sequence across the displayed sky-latitude range. This is a sky-position/velocity relation, not a conversion of $b$ into $Z$. Now generalize to the five published longitude slices.

In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(15, 8), sharex=True, sharey=True)
row_defs = [
    ("all rows", np.ones(len(stars13), dtype=bool)),
    (r"$V_R>0$", stars13["V_R"] > 0*u.km/u.s),
    (r"$V_R<0$", stars13["V_R"] < 0*u.km/u.s),
]
global_limit13 = float(np.max(np.abs(vr13)))
for column, lower in enumerate(range(150, 200, 10)):
    longitude = (stars13["l"] >= lower*u.deg) & (stars13["l"] < (lower+10)*u.deg)
    for row, (label, sign_mask) in enumerate(row_defs):
        selected = longitude & sign_mask
        ax = axes[row, column]
        kwargs = dict(s=5, alpha=0.35, linewidths=0, rasterized=True)
        if row == 0:
            kwargs.update(c=vr13[selected], cmap="coolwarm",
                          norm=plt.Normalize(-global_limit13, global_limit13))
        else:
            kwargs.update(color="tab:red" if row == 1 else "tab:blue")
        ax.scatter(b13[selected], vz13[selected], **kwargs)
        ax.axhline(0, color="0.4", lw=0.6)
        if row == 0:
            ax.set_title("{}–{} deg".format(lower, lower+10))
        if column == 0:
            ax.set_ylabel(label + "\n" + r"$V_Z$ [km s$^{-1}$]")
        if row == 2:
            ax.set_xlabel(r"$b$ [deg]")
        ax.set(xlim=(20,40), ylim=(-180,180))
fig.suptitle("Figure 13: all rows → outward → inward", y=0.99)
fig.tight_layout()
plt.show()

## ⚠️ What this still does not prove

The table lacks upstream selection columns, distances, weights, uncertainties, identifiers, and the fitted ACS boundary. Connecting the sequence to the named photometric ACS overdensity requires the paper's unreleased definition. No new significance, membership, or causal claim is made. Marginal histograms are omitted because the lesson is joint $(b,V_Z)$ structure.

# Figure 9 — Recognizing a wave

## 👀 What am I supposed to see?

A wave has crests, troughs, zero crossings, and wavelength. Broad structure is more defensible here than naming every fluctuation.

## 🧹 Remove information, then return to the release

Start with ideal anatomy; then show all 70 released summaries and uncertainties without smoothing.

In [ ]:
phase = np.linspace(0, 2*np.pi, 500)
wave9 = load_figure9_wave(ROOT)
lz9 = wave9.lz.to_value(u.kpc*u.km/u.s)
vr9 = wave9.vr.to_value(u.km/u.s)
error9 = wave9.vr_uncertainty.to_value(u.km/u.s)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.3))
axes[0].plot(phase, np.sin(phase), color="black")
axes[0].axhline(0, color="0.5", lw=0.8)
for text, point, offset in [
    ("crest", (np.pi/2,1), (1.0,1.25)),
    ("trough", (3*np.pi/2,-1), (4.0,-1.3)),
    ("zero crossing", (np.pi,0), (2.2,0.45)),
]:
    axes[0].annotate(text, point, offset, arrowprops={"arrowstyle":"->"})
axes[0].set(title="Ideal sine-wave anatomy", xlabel="phase", ylabel="amplitude",
            ylim=(-1.45,1.4))
axes[1].errorbar(lz9, vr9, yerr=error9, fmt="o-", ms=3, lw=1, capsize=2)
axes[1].axhline(0, color="0.5", lw=0.8)
axes[1].axvspan(3100,3300,color="tab:blue",alpha=0.08,label="broad trough")
axes[1].axvspan(3600,3800,color="tab:blue",alpha=0.08)
axes[1].axvspan(4000,4100,color="tab:red",alpha=0.08,label="strong crest")
axes[1].set(title="Released Figure 9 (binned summaries, not stars)",
            xlabel=r"$L_Z$ [kpc km s$^{-1}$]",
            ylabel=r"released $V_R$ statistic [km s$^{-1}$]")
axes[1].legend(fontsize=8)
fig.tight_layout()
plt.show()

## 🔍 Isolate and 📏 transform

A seven-bin, order-two Savitzky–Golay line is shown **for the eye only** and never enters a spectrum. The synthetic lower views contain exactly three cycles uniformly in $x=1/L_Z$; only the horizontal coordinate changes. The reciprocal transformation both stretches/compresses spacing and reverses ordering on increasing left-to-right axes: high $L_Z$ corresponds to low $1/L_Z$, and low $L_Z$ to high $1/L_Z$. Rugs expose the transformed nonuniform sampling.

In [ ]:
smooth9_for_eye_only = savgol_filter(vr9, 7, 2)
x9 = 1/lz9
toy9 = np.sin(2*np.pi*3*(x9-x9.min())/np.ptp(x9))
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
axes[0].errorbar(lz9, vr9, yerr=error9, fmt="o", ms=3, alpha=0.5)
axes[0].plot(lz9, smooth9_for_eye_only, color="black", lw=2,
             label="7-bin smoother: for the eye only")
axes[0].legend(fontsize=8)
axes[0].set(title="Optional visual guide", xlabel=r"$L_Z$", ylabel=r"$V_R$")
order_lz = np.argsort(lz9)
axes[1].plot(lz9[order_lz], toy9[order_lz], "o-", ms=3)
axes[1].plot(lz9, np.full_like(lz9,-1.22), "|", color="0.3")
axes[1].text(0.98, 0.05, r"high $L_Z$ $\leftrightarrow$ low $1/L_Z$",
             transform=axes[1].transAxes, ha="right", va="bottom", fontsize=8)
axes[1].set(title="Same toy wave in $L_Z$", xlabel="uniformly sampled $L_Z$",
            ylabel="synthetic amplitude", ylim=(-1.35,1.2))
order_x = np.argsort(x9)
axes[2].plot(x9[order_x], toy9[order_x], "o-", ms=3)
axes[2].plot(x9, np.full_like(x9,-1.22), "|", color="0.3")
axes[2].text(0.02, 0.05, r"low $L_Z$ $\leftrightarrow$ high $1/L_Z$",
             transform=axes[2].transAxes, ha="left", va="bottom", fontsize=8)
axes[2].set(title="Equal wavelength in $x=1/L_Z$",
            xlabel=r"$1/L_Z$ [(kpc km s$^{-1}$)$^{-1}$]",
            ylabel="synthetic amplitude", ylim=(-1.35,1.2))
fig.tight_layout()
plt.show()

## 🧠 Rebuild and ⚠️ stop

The coordinate transformation changes spacing, not measured velocities. The released table contains summaries, not stars, and has documented statistic/bin-count ambiguities. Smoothing does not measure extrema or perturbation count. Treating $1/L_Z$ as privileged is model-dependent.

# Figure 10 — Frequency means wiggles per baseline

## 👀 What am I supposed to see?

More wiggles across the same support means higher frequency.

## 🧹 Remove information: noiseless toys

One, three, and six cycles land on discrete Fourier bins. The compact 2.6-cycle example shows finite non-integer support leaking into neighbouring bins; it is not another Lambert analysis.

In [ ]:
toy_cycles = [1.0, 3.0, 6.0, 2.6]
fig, axes = plt.subplots(4, 2, figsize=(11, 9),
                         gridspec_kw={"width_ratios":[1.5,1]})
for row, cycles_value in enumerate(toy_cycles):
    coordinate, signal, frequency, power = toy_cycle_spectrum(cycles_value)
    axes[row,0].plot(coordinate, signal, color="black")
    axes[row,0].set(ylabel=f"{cycles_value:g} cycles", xlim=(0,1), ylim=(-1.15,1.15))
    axes[row,1].stem(frequency[:12], (power/power.max())[:12],
                     basefmt=" ", linefmt="tab:purple", markerfmt="o")
    axes[row,1].set(xlim=(0,10), ylim=(0,1.08), ylabel="relative power")
    if cycles_value == 2.6:
        axes[row,1].set_title("non-integer support: leakage", fontsize=9)
        axes[row,1].annotate("one underlying frequency → power spread\nacross several discrete FFT bins",
                             xy=(3, 1), xytext=(5.1, 0.64), fontsize=8,
                             arrowprops={"arrowstyle": "->"})
axes[-1,0].set_xlabel("same unit baseline")
axes[-1,1].set_xlabel("cycles per baseline")
axes[0,0].set_title("synthetic waves")
axes[0,1].set_title("discrete spectra")
fig.tight_layout()
plt.show()

## 🔍 Isolate the real support and 🧠 rebuild

Use $N_{\mathrm{cycles}}=f\,\Delta x_{\mathrm{support}}$. Cycle counts are derived from Figure 9 at runtime. Figure 10 is displayed directly; Notebook 3's forensic FFT audit is not repeated.

In [ ]:
spectrum10 = load_figure10_spectrum(ROOT)
published_frequencies = np.array([1313.0, 5878.6])
x_span10, resolution10, cycles10 = spectral_resolution(x9, published_frequencies)
display(Table([published_frequencies, cycles10],
    names=["published fitted frequency [kpc km/s]", "cycles over released support"]))
print(f"x support = {x_span10:.12e}; 1/support = {resolution10:.1f} kpc km/s")
frequency10 = spectrum10.frequency.to_value(u.kpc*u.km/u.s)
scale10 = spectrum10.power.max()
order10 = np.argsort(frequency10)
fig, ax = plt.subplots(figsize=(9,4.6))
ax.plot(frequency10[order10], (spectrum10.power/scale10)[order10], "o-", ms=4,
        label="released power / maximum")
ax.fill_between(frequency10[order10],
    ((spectrum10.power-spectrum10.power_spread)/scale10)[order10],
    ((spectrum10.power+spectrum10.power_spread)/scale10)[order10],
    alpha=0.15, label="released spread / same maximum")
for value, count in zip(published_frequencies, cycles10):
    ax.axvline(value, color="0.25", ls="--")
    ax.text(value, 1.02, f"{count:.2f} cycles", rotation=90,
            ha="center", va="bottom", fontsize=8)
secondary = ax.secondary_xaxis("top", functions=(
    lambda frequency: frequency * x_span10,
    lambda cycle_count: cycle_count / x_span10,
))
secondary.set_xlabel("cycles across released Figure-9 support")
ax.set(title="Released Figure 10: frequency versus finite support",
       xlabel=r"frequency [kpc km s$^{-1}$]", ylabel="released relative power")
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

## ⚠️ What this still does not prove

A fine numerical grid is not fine physical resolution. The published frequencies span about 0.58 and 2.60 cycles over the released support. Author resampling, normalization, Monte Carlo aggregation, and Gaussian fitting remain unavailable. No frequency is tuned here and no spectrum identifies a unique perturber.

# Final checklist

Ask what is released, what was removed, whether a 1-D reduction survives return to the full plot, whether a guide is being mistaken for a fit, and where data-supported inference ends. Simplification reveals structure; it cannot create missing source data, uncertainties, selection metadata, or causal evidence.

In [ ]:
np.testing.assert_allclose(longitude5[FIG5_COLUMNS[[0,-1]]],
                           [175.8333333333,184.1666666667])
assert counts13 == (1228,572,656)
np.testing.assert_array_equal(guide13.counts, [73,72,76,85,63,59,54,55,23,12])
np.testing.assert_allclose([row[3] for row in peak_rows],
                           [235.46875,219.21875,219.21875,194.84375])
assert np.allclose(np.diff(lz9), np.diff(lz9)[0], rtol=1e-12)
assert not np.allclose(np.diff(x9), np.diff(x9)[0], rtol=1e-3, atol=0)
NOTEBOOK_STAGE = "task-6b-complete"
print("Notebook 4 completed offline:", NOTEBOOK_STAGE)